# Niche membership: language mix vs. regional circuit, and audience overlap within genre×platform

**Purpose:** the brief's whole framework rests on treating a genre×platform
(×region) cell as a "niche" — a shared audience pool where competitive
exclusion could apply. This notebook checks that assumption two ways,
using `language_mix_snapshots` (Twitch broadcast-language mix, the region
proxy described in PRD §6) as a stand-in for "who's actually watching":

1. **Does each title's audience-language profile agree with its
   competitive circuit's regional footprint?** (`tournaments.region`,
   tier-1 events only.) A title whose circuit is global but whose Twitch
   audience is one language, or the reverse, is informative either way —
   it says something about whether the competitive structure and the
   fandom are actually the same population.
2. **Do titles sharing a genre×platform cell actually share an audience?**
   Computed as language-distribution similarity between every pair of
   titles in the same cell (PUBG Mobile vs. Free Fire, Wild Rift vs.
   Mobile Legends: Bang Bang, League vs. Dota 2, and every other
   same-cell pair, not just those three examples) — this is a direct
   test of whether "same niche cell" means "same audience pool" in
   practice, or just "same genre label."

**This is a ~3-4 day snapshot (2026-08-31 through 2026-09-04 as of this
run — re-check the printed range below, since re-running this notebook
later will pick up whatever `language_mix_snapshots` has accumulated by
then), not a mature baseline.** Everything below is directional — worth
re-running once `language_mix_snapshots` has weeks or months of history,
not treated as a settled answer from this alone. The contamination check
right below exists because a snapshot this short is exactly the kind of
window a single live event can dominate.

In [1]:
# Imports and repo path setup — same pattern as the other notebooks.
import sys
from itertools import combinations
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import yaml

from analysis.metrics import _normalize_tier
from etl.db import get_connection

In [2]:
# All 23 active titles, with the Phase 3 genre/platform tags (committed
# 2026-09-05) — config/titles.yaml is the source of truth for these, same
# as every other notebook in this project.
with open(REPO_ROOT / "config" / "titles.yaml") as f:
    titles_config = yaml.safe_load(f)["titles"]

active_titles = [t for t in titles_config if t.get("is_active")]
title_ids = sorted(t["id"] for t in active_titles)
display_name_by_id = {t["id"]: t["display_name"] for t in active_titles}
genre_platform_by_id = {t["id"]: (t.get("genre"), t.get("platform")) for t in active_titles}

len(title_ids), title_ids[:5]

(23,
 ['age_of_empires_ii',
  'apex_legends',
  'brawl_stars',
  'counter_strike',
  'dota2'])

## Contamination check — first, per the reason this notebook exists

A live tournament running during the snapshot window pulls language mix
toward its broadcast's host region/language (official-broadcast viewers,
co-streamers in the host country's language) — over a full year that
averages out; over 3-4 days it can dominate. Checked directly against
`tournaments.start_date`/`end_date` for the exact window
`language_mix_snapshots` covers, not assumed clean.

Three severity tiers, not a single yes/no: a **tier-1** event overlapping
is the highest-risk case (biggest broadcasts, most likely to visibly skew
a short window); a **tier-2-or-below-only** overlap is lower risk (still
real, but these are usually one of several parallel regional leagues
running continuously, not a single concentrated spike); **no overlap**
is the cleanest case. "Tier-1" here uses the same `_normalize_tier`
letter-tier fix from this session's reconciliation work (S-Tier counts as
tier-1), for the same reason it mattered there — VALORANT and
Counter-Strike would otherwise show zero tier-1 events even when one is
actually running.

In [3]:
conn = get_connection()
window_start, window_end = conn.execute(
    "SELECT MIN(captured_at), MAX(captured_at) FROM language_mix_snapshots"
).fetchone()
print(f"language_mix_snapshots window: {window_start} to {window_end}")

overlap_rows = conn.execute(
    """
    SELECT title_id, liquipedia_page, tier, region, start_date, end_date
    FROM tournaments
    WHERE start_date IS NOT NULL AND end_date IS NOT NULL
      AND date(start_date) <= date(?) AND date(end_date) >= date(?)
    """,
    (window_end[:10], window_start[:10]),
).fetchall()
conn.close()

overlap_df = pd.DataFrame(overlap_rows, columns=["title_id", "page", "tier", "region", "start_date", "end_date"])
overlap_df["tier_normalized"] = overlap_df["tier"].apply(_normalize_tier)

tier1_overlap = set(overlap_df.loc[overlap_df["tier_normalized"] == "1", "title_id"])
any_overlap = set(overlap_df["title_id"])
tier2_only_overlap = any_overlap - tier1_overlap
no_overlap = set(title_ids) - any_overlap

contamination = pd.DataFrame(
    {
        "title_id": title_ids,
        "display_name": [display_name_by_id[t] for t in title_ids],
        "overlap_severity": [
            "TIER-1 EVENT LIVE" if t in tier1_overlap
            else "tier-2-or-below only" if t in tier2_only_overlap
            else "no overlap"
            for t in title_ids
        ],
        "overlapping_tournaments": [
            "; ".join(overlap_df.loc[overlap_df["title_id"] == t, "page"]) if t in any_overlap else ""
            for t in title_ids
        ],
    }
).sort_values("overlap_severity")

print(f"\n{len(tier1_overlap)}/23 titles: TIER-1 event live during the window — highest skew risk")
print(f"{len(tier2_only_overlap)}/23 titles: only tier-2-or-below overlap")
print(f"{len(no_overlap)}/23 titles: no overlap — cleanest baseline")
contamination

language_mix_snapshots window: 2026-08-31T10:13:48Z to 2026-09-04T09:25:58Z

10/23 titles: TIER-1 event live during the window — highest skew risk
6/23 titles: only tier-2-or-below overlap
7/23 titles: no overlap — cleanest baseline


,title_id,display_name,overlap_severity,overlapping_tournaments
0,age_of_empires_ii,Age of Empires II,TIER-1 EVENT LIVE,The League; The League/Division 2
20,tekken,Tekken 8,TIER-1 EVENT LIVE,Takedown/2026/T8
19,teamfight_tactics,Teamfight Tactics,TIER-1 EVENT LIVE,Enchanted Wilds/TFT Pro Circuit/AMER/Riftbeast...
3,counter_strike,Counter-Strike 2,TIER-1 EVENT LIVE,BLAST/Open/2026/Fall; BLAST/Premier/2026/Frequ...
18,street_fighter,Street Fighter 6,TIER-1 EVENT LIVE,Street Fighter League/EMEAA/2026; Street Fight...
16,rocket_league,Rocket League,TIER-1 EVENT LIVE,FIFAe World Cup/Continental Championships/2026...
15,rainbow_six_siege,Rainbow Six Siege,TIER-1 EVENT LIVE,Asia Pacific League/2026/APAC North; Asia Paci...
12,overwatch,Overwatch,TIER-1 EVENT LIVE,Overwatch Champions Series/2026/China/Stage 3/...
21,valorant,VALORANT,TIER-1 EVENT LIVE,China Evolution Series/2026/Act 3; VCT/2026/Am...
9,league_of_legends,League of Legends,TIER-1 EVENT LIVE,CBLOL/2026/Split 2; Circuito Desafiante/2026/S...


## Language distribution per title

`language_mix_snapshots` aggregates viewer_count by `(title_id,
captured_at, language_code)` at poll time (`collectors/twitch_poll.py`) —
summing `viewer_count` across the whole window per `(title_id,
language_code)` and normalizing to a share within each title gives a
language distribution weighted by actual viewer-time, not just by how
many polls happened to catch a given language.

In [4]:
conn = get_connection()
lang_rows = conn.execute(
    "SELECT title_id, language_code, SUM(viewer_count) AS total_viewer_time "
    "FROM language_mix_snapshots GROUP BY title_id, language_code"
).fetchall()
conn.close()

lang_df = pd.DataFrame(lang_rows, columns=["title_id", "language_code", "total_viewer_time"])
title_totals = lang_df.groupby("title_id")["total_viewer_time"].transform("sum")
lang_df["share"] = lang_df["total_viewer_time"] / title_totals

# Wide pivot for a readable per-title top-languages view — top 5 languages
# by total share across all titles, "other" collapsing the rest, so the
# table stays legible rather than one column per language code seen.
top_languages = (
    lang_df.groupby("language_code")["total_viewer_time"].sum().sort_values(ascending=False).head(8).index.tolist()
)
lang_df["language_bucket"] = lang_df["language_code"].where(lang_df["language_code"].isin(top_languages), "other")
lang_pivot = (
    lang_df.groupby(["title_id", "language_bucket"])["share"].sum().unstack(fill_value=0.0)
    .reindex(columns=top_languages + ["other"], fill_value=0.0)
)
lang_pivot.index = [display_name_by_id[t] for t in lang_pivot.index]
(lang_pivot * 100).round(1)

language_bucket,en,ru,ja,fr,es,zh,pt,de,other
Age of Empires II,81.7,1.7,0.0,7.4,3.9,0.4,0.8,2.4,1.7
Apex Legends,47.7,2.8,36.0,2.4,1.5,5.9,0.4,1.5,1.8
Brawl Stars,15.6,27.0,0.0,7.4,34.6,0.0,0.3,6.9,8.1
Counter-Strike 2,24.0,46.4,0.1,3.0,2.7,0.1,9.9,2.1,11.7
Dota 2,25.9,66.3,0.0,0.0,0.5,0.0,4.3,0.3,2.6
Fortnite,65.2,2.7,0.3,4.4,9.4,0.0,4.9,5.5,7.5
Free Fire,3.8,0.2,0.4,1.9,73.8,0.0,19.2,0.0,0.6
Guilty Gear -Strive-,80.0,1.9,8.5,2.9,3.1,0.6,0.4,0.0,2.6
Hearthstone,52.6,22.3,2.8,11.7,1.0,3.2,2.0,1.4,2.8
League of Legends,43.5,2.5,6.4,11.9,8.4,4.3,4.9,9.1,9.1


In [5]:
# Sample size matters here: a title with a small total Twitch viewer-time
# base over this window has its language mix much more easily dominated by
# one or two individual streamers, not a representative audience read.
# Confirmed concretely below, not just in principle — Mobile Legends: Bang
# Bang's total is ~60K vs. League of Legends' ~2.0M (33x smaller), and MLBB
# shows 88.9% Russian-language share, which does not match its known
# real-world Southeast-Asian-majority player base. The likely mechanism:
# MLBB's Twitch presence is thin (its actual audience is concentrated on
# other platforms in SEA), so this window's handful of MLBB Twitch streams
# happened to skew toward one or two CIS streamers -- not a representative
# sample of MLBB's real audience. Treat any title flagged "low sample" here
# with extra caution in the AGREE/DIVERGE and similarity results below.
sample_size = lang_df.groupby("title_id")["total_viewer_time"].sum().sort_values()
sample_size_display = sample_size.rename("total_viewer_time").reset_index()
sample_size_display["title"] = sample_size_display["title_id"].map(display_name_by_id)
sample_size_display["low_sample_flag"] = sample_size_display["total_viewer_time"] < sample_size.median() / 5
sample_size_display[["title", "total_viewer_time", "low_sample_flag"]]

,title,total_viewer_time,low_sample_flag
0,Guilty Gear -Strive-,3647,True
1,Free Fire,4448,True
2,Mortal Kombat 1,14072,True
3,League of Legends: Wild Rift,16394,True
4,Brawl Stars,24226,True
5,PUBG Mobile,25316,True
6,StarCraft II,26232,True
7,Tekken 8,39766,True
8,Age of Empires II,50928,False
9,Mobile Legends: Bang Bang,60453,False


## Regional profile from tier-1 tournaments

Distribution of each title's **tier-1** tournaments (normalized, as
above) across `tournaments.region` — count-based (one tournament, one
vote), not weighted by prize pool or attendance. `region` is null for
some tournaments (country not in `collectors/liquipedia.py`'s
`COUNTRY_TO_REGION` lookup, or country itself missing) — those are kept
as an explicit "unknown" share rather than silently dropped from the
denominator.

In [6]:
conn = get_connection()
tier_rows = conn.execute("SELECT title_id, tier, region FROM tournaments").fetchall()
conn.close()

tier_df = pd.DataFrame(tier_rows, columns=["title_id", "tier", "region"])
tier_df["tier_normalized"] = tier_df["tier"].apply(_normalize_tier)
tier1_df = tier_df[tier_df["tier_normalized"] == "1"].copy()
tier1_df["region"] = tier1_df["region"].fillna("unknown")

tier1_counts = tier1_df.groupby("title_id")["title_id"].transform("count")
tier1_df["share"] = 1 / tier1_counts

region_profile = (
    tier1_df.groupby(["title_id", "region"])["share"].sum().unstack(fill_value=0.0).reindex(index=title_ids, fill_value=0.0)
)
tier1_tournament_count = tier1_df.groupby("title_id").size().reindex(title_ids, fill_value=0)

no_tier1 = [t for t in title_ids if tier1_tournament_count[t] == 0]
print(f"Titles with zero tier-1 tournaments in tournaments (no regional profile possible): {[display_name_by_id[t] for t in no_tier1]}")

region_profile_display = (region_profile * 100).round(1)
region_profile_display.insert(0, "tier1_tournament_count", tier1_tournament_count)
region_profile_display.index = [display_name_by_id[t] for t in region_profile_display.index]
region_profile_display

Titles with zero tier-1 tournaments in tournaments (no regional profile possible): []


region,tier1_tournament_count,Africa,Asia,Europe,North America,Oceania,South America,unknown
Age of Empires II,220,0.0,6.4,7.3,0.5,0.0,0.9,85.0
Apex Legends,93,0.0,20.4,10.8,9.7,0.0,0.0,59.1
Brawl Stars,18,0.0,16.7,27.8,0.0,0.0,5.6,50.0
Counter-Strike 2,489,0.0,11.5,40.1,13.9,1.2,2.2,31.1
Dota 2,1672,0.1,19.2,7.7,3.0,0.0,0.6,69.5
Fortnite,332,0.0,1.5,2.7,6.9,0.6,12.7,75.6
Free Fire,35,0.0,54.3,2.9,0.0,0.0,8.6,34.3
Guilty Gear -Strive-,65,0.0,26.2,20.0,49.2,0.0,3.1,1.5
Hearthstone,2754,0.0,2.8,1.3,1.0,0.0,0.0,94.8
League of Legends,314,0.0,30.6,20.1,17.5,0.3,4.5,27.1


## Where language mix and regional circuit agree or diverge

Comparing these needs a shared basis — `language_mix_snapshots` is coded
by language, `tournaments.region` by continent. Bridged with a rough
language→region proxy, in the same spirit as PRD §6's own description of
this exact tradeoff ("Korean or Japanese imply a region with reasonable
confidence; English spans North America, the UK, India and the
Philippines; Spanish spans Spain and most of Latin America"). Built
narrowly on purpose: only languages with one clearly dominant Twitch
region get mapped; **`en` and `es` are deliberately left unmapped** (not
guessed) since they're each split across regions this project already
treats as distinct — folding them in would manufacture false agreement or
false divergence. A title whose language mix is mostly `en`/`es` gets an
explicit "not regionalizable from language alone" read rather than a
forced call.

`AGREE` / `DIVERGE` compares each title's **top** language-implied region
against its **top** tier-1-tournament region — a coarse binary read, not
a distributional distance, because the point here is "do these two
signals point the same direction," not "by how much."

In [7]:
# Rough language -> region proxy. Deliberately incomplete, matching
# collectors/liquipedia.py's COUNTRY_TO_REGION convention: covers
# languages actually seen in this project's data with a reasonably
# confident single dominant region; en/es excluded on purpose (see markdown
# above). Region labels match tournaments.region exactly so the two sides
# are directly comparable.
LANGUAGE_TO_REGION = {
    "ko": "Asia", "ja": "Asia", "zh": "Asia", "th": "Asia", "vi": "Asia",
    "id": "Asia", "tl": "Asia", "hi": "Asia", "ms": "Asia", "ar": "Asia", "he": "Asia",
    "ru": "Europe", "de": "Europe", "fr": "Europe", "it": "Europe", "pl": "Europe",
    "tr": "Europe", "uk": "Europe", "cs": "Europe", "hu": "Europe", "el": "Europe",
    "fi": "Europe", "sv": "Europe", "da": "Europe", "no": "Europe", "nl": "Europe",
    "ro": "Europe", "bg": "Europe", "sk": "Europe", "hr": "Europe",
    "pt": "South America",  # medium confidence: Twitch's Portuguese-speaking base skews Brazil, but Portugal exists too
}
AMBIGUOUS_LANGUAGES = {"en", "es", "other"}

rows = []
for title_id in title_ids:
    title_langs = lang_df[lang_df["title_id"] == title_id]
    mapped = title_langs[title_langs["language_code"].isin(LANGUAGE_TO_REGION)].copy()
    mapped["implied_region"] = mapped["language_code"].map(LANGUAGE_TO_REGION)
    region_shares = mapped.groupby("implied_region")["share"].sum()
    mapped_total = region_shares.sum()

    top_tournament_region = None
    if tier1_tournament_count.get(title_id, 0) > 0:
        row = region_profile.loc[title_id]
        top_tournament_region = row.idxmax() if row.max() > 0 else None

    if mapped_total < 0.15:  # too little regionalizable language signal to call
        status, top_lang_region = "not regionalizable from language (mostly en/es/other)", None
    else:
        top_lang_region = region_shares.idxmax()
        if top_tournament_region is None:
            status = "no tier-1 tournament data to compare against"
        elif top_tournament_region == "unknown":
            status = "tournament region unmapped — can't compare"
        elif top_lang_region == top_tournament_region:
            status = "AGREE"
        else:
            status = "DIVERGE"

    rows.append({
        "title": display_name_by_id[title_id],
        "top_language_implied_region": top_lang_region,
        "mapped_language_share_pct": round(mapped_total * 100, 1),
        "top_tier1_tournament_region": top_tournament_region,
        "status": status,
    })

agreement_df = pd.DataFrame(rows)
print(agreement_df["status"].value_counts())
agreement_df.sort_values("status")

status
tournament region unmapped — can't compare               11
DIVERGE                                                   6
not regionalizable from language (mostly en/es/other)     3
AGREE                                                     3
Name: count, dtype: int64


,title,top_language_implied_region,mapped_language_share_pct,top_tier1_tournament_region,status
18,Street Fighter 6,Asia,84.4,Asia,AGREE
3,Counter-Strike 2,Europe,73.2,Europe,AGREE
21,VALORANT,Asia,67.2,Asia,AGREE
22,League of Legends: Wild Rift,South America,60.2,Asia,DIVERGE
20,Tekken 8,Europe,17.0,Asia,DIVERGE
14,PUBG Mobile,Europe,92.2,Asia,DIVERGE
6,Free Fire,South America,22.3,Asia,DIVERGE
7,Guilty Gear -Strive-,Asia,16.8,North America,DIVERGE
9,League of Legends,Europe,47.8,Asia,DIVERGE
15,Rainbow Six Siege,NaN,12.7,unknown,not regionalizable from language (mostly en/es...


## The main question: do same-cell titles actually share an audience?

Every pair of titles sharing a `(genre, platform)` cell (Phase 3 tags),
not just the three example pairs — a `(genre, platform)` cell with only
one title has no pair to test and is skipped.

**Three metrics shown side by side, not one replacing another** — the
first version of this notebook used only plain cosine similarity over
full language-share vectors, and it was misleading: Apex Legends (36%
`ja`) vs. Fortnite (0.3% `ja`) scored 0.789, a real audience difference
compressed into a "these look similar" number, because both titles'
shares are dominated by the shared `en` axis (47.7% and 65.2%) and cosine
similarity is pulled toward whichever axis is largest in both vectors at
once. Three views instead:

- **`cosine_full`** — the original metric, kept for comparison, not
  hidden. Shows exactly the masking problem above.
- **`cosine_no_english`** — the same metric with the `en` column dropped
  from both vectors before computing. Directly answers "if we ignore the
  one language almost every title has some of, how similar is the rest
  of the audience?" — this is the metric that actually surfaces the
  Apex/Fortnite `ja` gap.
- **`jensen_shannon_distance`** — a genuinely different metric on the
  full distribution (not just cosine with a dimension removed): the
  square root of Jensen-Shannon divergence, bounded [0, 1], a proper
  distance between two probability distributions. **This one runs the
  opposite direction from the cosine columns** — 0 means identical, 1
  means completely disjoint — flagged here so it isn't misread as another
  similarity score on the same scale.

None of the three is "the" answer; they're three different lenses on the
same question, shown together on purpose.

In [8]:
# Full (title x language_code) share matrix, not the top-8-bucketed
# display version above — every metric here should see every language
# each title actually has data for.
full_lang_matrix = lang_df.pivot(index="title_id", columns="language_code", values="share").fillna(0.0)
full_lang_matrix_no_en = full_lang_matrix.drop(columns=["en"], errors="ignore")


def cosine_similarity(a: pd.Series, b: pd.Series) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else float("nan")


def jensen_shannon_distance(p: pd.Series, q: pd.Series) -> float:
    """sqrt(JS divergence), base-2 log -> bounded [0, 1]. 0 = identical
    distributions, 1 = completely disjoint. A distance, not a similarity
    — runs the opposite direction from the cosine columns."""
    p, q = p.values, q.values
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / b[mask]))

    jsd = 0.5 * kl(p, m) + 0.5 * kl(q, m)
    return float(np.sqrt(max(jsd, 0.0)))


# Group by (genre, platform); only cells with 2+ titles produce a pair.
from collections import defaultdict

cells = defaultdict(list)
for title_id in title_ids:
    cells[genre_platform_by_id[title_id]].append(title_id)

# Mobile-platform pairs are flagged not interpretable below (fix per
# explicit instruction), based on the sample-size table above: every
# "mobile"-platform title in a same-cell pair here (Mobile Legends: Bang
# Bang, Wild Rift, PUBG Mobile, Free Fire) is low_sample_flag=True, and
# three of the four show a single-language share that contradicts their
# known real-world audience (MLBB 88.9% ru, PUBG Mobile 74.2% ru, Free
# Fire 73.8% es) — the mechanism is the same one confirmed for MLBB: thin
# Twitch-specific samples dominated by whichever few streamers were live,
# not a representative read of these titles' actual (largely non-Twitch)
# audiences.
pair_rows = []
for (genre, platform), members in cells.items():
    if len(members) < 2:
        continue
    for a, b in combinations(sorted(members), 2):
        row_a, row_b = full_lang_matrix.loc[a], full_lang_matrix.loc[b]
        row_a_ne, row_b_ne = full_lang_matrix_no_en.loc[a], full_lang_matrix_no_en.loc[b]
        interpretable = platform != "mobile"
        pair_rows.append({
            "genre": genre, "platform": platform,
            "title_a": display_name_by_id[a], "title_b": display_name_by_id[b],
            "cosine_full": round(cosine_similarity(row_a, row_b), 3),
            "cosine_no_english": round(cosine_similarity(row_a_ne, row_b_ne), 3),
            "jensen_shannon_distance": round(jensen_shannon_distance(row_a, row_b), 3),
            "interpretable_from_twitch_data": interpretable,
        })

pairs_df = pd.DataFrame(pair_rows).sort_values("cosine_no_english", ascending=False)
n_not_interpretable = (~pairs_df["interpretable_from_twitch_data"]).sum()
print(f"{len(pairs_df)} same-cell pair(s) across {sum(1 for m in cells.values() if len(m) >= 2)} multi-title (genre, platform) cell(s)")
print(f"{n_not_interpretable} pair(s) flagged NOT interpretable — mobile-platform titles, Twitch sample too thin to represent their real audience")
pairs_df

13 same-cell pair(s) across 7 multi-title (genre, platform) cell(s)
2 pair(s) flagged NOT interpretable — mobile-platform titles, Twitch sample too thin to represent their real audience


,genre,platform,title_a,title_b,cosine_full,cosine_no_english,jensen_shannon_distance,interpretable_from_twitch_data
0,rts,pc,Age of Empires II,StarCraft II,0.995,0.784,0.215,True
11,fighting,console,Mortal Kombat 1,Tekken 8,0.998,0.720,0.227,True
3,battle_royale,pc_console,Fortnite,PUBG: BATTLEGROUNDS,0.627,0.550,0.557,True
6,tac_fps,pc,Rainbow Six Siege,VALORANT,0.662,0.520,0.579,True
9,moba,mobile,Mobile Legends: Bang Bang,League of Legends: Wild Rift,0.407,0.437,0.653,False
12,fighting,console,Street Fighter 6,Tekken 8,0.228,0.319,0.749,True
2,battle_royale,pc_console,Apex Legends,PUBG: BATTLEGROUNDS,0.583,0.300,0.558,True
4,tac_fps,pc,Counter-Strike 2,Rainbow Six Siege,0.459,0.241,0.650,True
5,tac_fps,pc,Counter-Strike 2,VALORANT,0.415,0.190,0.641,True
7,moba,pc,Dota 2,League of Legends,0.386,0.145,0.711,True


### PUBG: BATTLEGROUNDS and Guilty Gear -Strive- — one fixed, one checked and kept

Both were singleton `(genre, platform)` cells in an earlier version of
this notebook, both for the same mechanical reason: a `platform` tag that
diverged from their nearest genre peers. Checked individually rather than
treated as the same issue, because they turned out not to be:

- **PUBG was reclassified 2026-09-08** from `pc` to `pc_console`
  (`config/titles.yaml`), joining Apex Legends and Fortnite's
  `battle_royale/pc_console` cell. The original `pc` tag had a real
  citation (PGC/PCS's premier tier runs PC-only), but on reflection that
  reflects a competitive-*format* choice, not a genuine audience-market
  split — unlike Apex/Fortnite, PUBG's `pc`-only tag was isolating it
  from its closest genre peers without a correspondingly strong
  audience-level reason. An explicit override, not a re-verification —
  the original citation is kept in the config comment for reference.
- **Guilty Gear -Strive- stays `pc_console`, checked for the same issue
  and kept as a singleton.** Its citation is a different *kind* of claim,
  not just the same claim about a different game: ARC World Tour has a
  genuine dual-platform qualification structure — a PS5 path *and* a
  separate PC path feeding the same Finals — which is structurally the
  same shape as Apex/Fortnite's own justification for `pc_console`
  (parallel platforms feeding one unified competition), not PUBG's
  original shape (one platform, full stop). Reclassifying Guilty Gear
  here would need Tekken/Street Fighter/Mortal Kombat's `console` tag to
  be the artifact instead — a different, unchecked claim not raised here.

## Caveats — read everything above as directional, not settled

- **This is a ~3-4 day snapshot.** `language_mix_snapshots` only started
  2026-08-31; every number above is a few days of hourly polls, not a
  seasonally-averaged baseline. Re-run this notebook once that table has
  weeks or months of history before treating any AGREE/DIVERGE call or
  similarity score as a finding rather than a first look.
- **Read every result against the contamination table above it.** A
  title marked `TIER-1 EVENT LIVE` had its language mix measured while a
  flagship broadcast was actively skewing it toward that event's
  host region/language — its AGREE/DIVERGE status and its pairwise
  similarity scores are the *most* likely to shift once the snapshot
  window lengthens.
- **Every mobile-platform pair's similarity scores are marked not
  interpretable, not just flagged as low-confidence.** Three of the four
  mobile-platform titles that appear in a same-cell pair show a
  single-language share that contradicts their known real-world audience
  — Mobile Legends: Bang Bang (88.9% Russian), PUBG Mobile (74.2%
  Russian), and Free Fire (73.8% Spanish) — and all four (these three
  plus Wild Rift) are `low_sample_flag=True` in the sample-size table
  above. The mechanism is the same in each case: these titles' real
  audiences aren't concentrated on Twitch, so this window's thin
  Twitch-specific sample was dominated by whichever few streamers
  happened to be live, not a representative read. The
  `interpretable_from_twitch_data` column reflects this directly rather
  than reporting a numeric score as if it were a normal result — treat
  Mobile Legends/Wild Rift and Free Fire/PUBG Mobile's numbers as "not
  measurable from this data," not as "low similarity."
- **The language→region proxy is deliberately narrow** (`en`/`es`
  excluded, PRD §6's own caveat) — a title flagged "not regionalizable
  from language" isn't a data gap, it's the proxy correctly declining to
  guess.
- **`tournaments.region` is unmapped for the majority of tier-1 events
  for many titles** (Hearthstone 94.8% unknown, Age of Empires II 85%,
  Fortnite 75.6% — see the region-profile table above), which is why 11
  of 23 titles land on "tournament region unmapped — can't compare"
  rather than an actual AGREE/DIVERGE call. This is an existing gap in
  `collectors/liquipedia.py`'s `COUNTRY_TO_REGION` coverage /
  infobox `country` field completeness, not something this notebook
  introduces or fixes — worth its own follow-up if the region-comparison
  side of this analysis matters more than the audience-similarity side.
- **None of the three pairwise metrics measures size or competitive
  standing** — a high `cosine_no_english` or a low
  `jensen_shannon_distance` says two titles' viewers look similar by
  language mix, not that they're equally successful or that one is
  winning a niche contest. That's a separate question from what this
  notebook checks.